# ADL DifferenceFusion v2 — ACTUAL SCORED COMPETITION RUN ONLY — FULL TRACE

**Uploadable build**

This version keeps the full scored-run logs but does **not** embed the ~45 MB
`generalized_action_model.pt` inside the `.ipynb`.

Required Kaggle inputs:
- ARC-AGI-3 competition input
- TAAF source-share
- vLLM wheelhouse
- Qwen 3.8 FP8
- `generalized_action_model` dataset containing `generalized_action_model.pt`

The notebook automatically finds:
`/kaggle/input/**/generalized_action_model.pt`

Runtime:
- actual Competition Rerun only
- one real trajectory per game
- every PLAN → ACTION → RESULT → POST_MOVE_ADL step visible
- no local/dummy submission path
- no synthetic score


In [ ]:

import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime
from pathlib import Path
from urllib.request import urlopen

NOTEBOOK_START_EPOCH = time.time()
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}

# HARD CONTRACT: this notebook is not allowed to run as a local/public evaluator.
if not TRUE_SUBMISSION:
    raise RuntimeError(
        "ADL DifferenceFusion v2 is competition-rerun-only. "
        "Run it through Kaggle's actual Competition Rerun; local/Save & Run All is intentionally disabled."
    )

os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "0"
os.environ["ONLY_RESET_LEVELS"] = "true"

# FULL TRACE: scored run must show each planning/action/learning step.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "0"
os.environ["TAAF_VERBOSE"] = "1"
os.environ["TAAF_FULL_TRACE"] = "1"
os.environ["TAAF_LOG_EVERY_STEP"] = "1"
os.environ["TAAF_SAVE_REQUEST_LOGS"] = "1"
os.environ["DIFFERENCEFUSION_FULL_TRACE"] = "1"


# No cross-game learned plans / route caches.
os.environ["SIGIL_USE_PRIOR_PLAN_CACHE"] = "0"
os.environ["LS20_USE_LEARNED_PLANS"] = "0"

cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry
    for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)]
    if entry
)

print("ADL DifferenceFusion v2")
print("MODE=ACTUAL_COMPETITION_RERUN_ONLY")
print("TRUE_SUBMISSION=", TRUE_SUBMISSION)


## 1. Install the official ARC runtime

In [ ]:

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)


## 2. Locate the TAAF source bundle and Qwen 3.8 model

In [ ]:

DATASET_SOURCES = [
    "jeroencottaar/taaf-kaggle-source-share",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
    "saltb0x/qwen3-8-27b-fp8",
]
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"

def _find_bundle_dir() -> Path:
    hits = list(Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER))
    if not hits:
        raise RuntimeError("TAAF source bundle not found under /kaggle/input.")
    return hits[0].parent

def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [
        Path("/kaggle/input") / slug,
        Path("/kaggle/input/datasets") / owner / slug,
    ]

def _first_existing(candidates):
    return next((c for c in candidates if c.exists()), None)

BUNDLE_DIR = _find_bundle_dir()
kaggle_input_paths = {}

for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": "[]",
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")

print("source bundle:", BUNDLE_DIR)
print("inputs:", setup_env["TAAF_KAGGLE_INPUT_PATHS"])


## 3. Import bundled source and start Qwen 3.8

In [ ]:

def _source_path_entries(bundle_dir: Path):
    entries = []
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        raise RuntimeError(f"Bundle has no src directory: {src_root}")
    for repo in sorted(src_root.iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries

def _command_env():
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    if SETUP_ENV_PATH.exists():
        env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env

source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))

QWEN38_DATASET = "saltb0x/qwen3-8-27b-fp8"
QWEN38_OWNER = "saltb0x"
QWEN38_SLUG = "qwen3-8-27b-fp8"
QWEN38_SERVED_MODEL = "saltb0x/Qwen3.8-27B-FP8"

def _qwen38_setup_command(command: str) -> str:
    replacements = {
        "MODEL_OWNER = 'driessmit1'": f"MODEL_OWNER = '{QWEN38_OWNER}'",
        "MODEL_SLUG = 'vrfai-qwen3-6-27b-fp8-hf-snapshot'": f"MODEL_SLUG = '{QWEN38_SLUG}'",
        "SERVED_MODEL_NAME = 'vrfai/Qwen3.6-27B-FP8'": f"SERVED_MODEL_NAME = '{QWEN38_SERVED_MODEL}'",
    }
    for old, new in replacements.items():
        if old not in command:
            raise RuntimeError(f"Qwen 3.8 preflight missing expected setup text: {old}")
        command = command.replace(old, new, 1)

    if "qwen3_coder" not in command or "--reasoning-parser" not in command:
        raise RuntimeError("Released Qwen tool/reasoning parsers are missing.")
    return command

qwen_mount = Path(kaggle_input_paths[QWEN38_DATASET])
if not qwen_mount.exists():
    raise FileNotFoundError(f"Qwen 3.8 dataset mount missing: {qwen_mount}")

env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    command = _qwen38_setup_command(command)
    print("SETUP:", command, flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    env = _command_env()
    os.environ.update(env)

for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)


## 4. Load the actual benchmark deployment target

In [ ]:

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as f:
    target = pickle.load(f)
target.actual_run_as_submission = True
target.is_competition_rerun = True

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as f:
    bm = pickle.load(f)
bm.job_dir = WORKING_DIR

print("solver=", type(bm.solver).__name__)
print("model=", getattr(bm.solver, "model", None))

# Keep analyzer/tool interactions available for the full scored trace.
if hasattr(bm.solver, "save_request_logs"):
    bm.solver.save_request_logs = True
print("[DIFFERENCEFUSION][TRACE] save_request_logs=", getattr(bm.solver, "save_request_logs", None), flush=True)


## 5. Install ADL v12 efficiency/recovery guards

In [ ]:

STRICT_NO_PRIOR = True
TARGET_CONCURRENCY = 4

bm.solver.concurrency = TARGET_CONCURRENCY

try:
    from taaf_grafts.composite import install as _install_taaf_grafts
except ModuleNotFoundError:
    _install_taaf_grafts = None

_graft_flags = {
    "shortcircuit": True,
    "efficiency": True,
    "retry_guard": True,
    "recovery": True,
    "context_window": int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768")),
}

if _install_taaf_grafts is not None:
    _install_taaf_grafts(bm, _graft_flags, expected_version=1)
else:
    print("taaf_grafts unavailable; base solver retained.", flush=True)

assert bm.solver.concurrency == TARGET_CONCURRENCY
assert "banking" not in _graft_flags
assert "transfer" not in _graft_flags

print(
    "DIFFERENCEFUSION GUARDS "
    f"concurrency={TARGET_CONCURRENCY} "
    f"flags={_graft_flags}",
    flush=True,
)


## 6. Generalized neural action critic

The checkpoint is discovered by filename under `/kaggle/input`, so it may be attached through any Kaggle dataset.

The 26-dimensional action representation is:

`8 action-ID one-hot + normalized x + normalized y + 16 clicked-symbol one-hot`

For non-click actions, coordinate and clicked-symbol fields are zero.


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def _find_generalized_action_model() -> Path:
    hits = list(Path("/kaggle/input").rglob("generalized_action_model.pt"))
    if len(hits) != 1:
        raise RuntimeError(
            "Attach the Kaggle dataset `generalized_action_model` containing "
            "`generalized_action_model.pt`. Expected exactly one matching file; "
            f"found {len(hits)}: {hits[:8]}"
        )
    return hits[0]


ACTION_MODEL_PATH = _find_generalized_action_model()

class _BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = x
        x = F.relu(self.bn1(self.conv1(x)), inplace=False)
        x = self.bn2(self.conv2(x))
        if self.downsample is not None:
            identity = self.downsample(identity)
        return F.relu(x + identity, inplace=False)

class GeneralizedActionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.grid_symbol_embedding = nn.Embedding(16, 16)
        self.stem = nn.Sequential(
            nn.Conv2d(16, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        self.layer1 = self._make_layer(64, 64, 2, 1)
        self.layer2 = self._make_layer(64, 128, 2, 2)
        self.layer3 = self._make_layer(128, 256, 2, 2)
        self.layer4 = self._make_layer(256, 512, 2, 2)
        self.state_fc = nn.Sequential(nn.Linear(512, 64), nn.ReLU())
        self.action_fc = nn.Sequential(nn.Linear(26, 64), nn.ReLU())
        self.head_fc = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))

    def _make_layer(self, in_ch, out_ch, blocks, stride):
        layers = [_BasicBlock(in_ch, out_ch, stride)]
        for _ in range(1, blocks):
            layers.append(_BasicBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)

    def forward(self, grid, action_features):
        # grid: B,H,W integer ARC symbols
        x = self.grid_symbol_embedding(grid.long().clamp(0, 15))  # B,H,W,16
        x = x.permute(0, 3, 1, 2).contiguous()
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        s = self.state_fc(x)
        a = self.action_fc(action_features.float())
        return self.head_fc(torch.cat([s, a], dim=1)).squeeze(-1)

def encode_action26(action_id: int, grid=None, x=None, y=None):
    vec = np.zeros(26, dtype=np.float32)
    action_id = int(action_id)
    if not 0 <= action_id <= 7:
        raise ValueError(f"action_id must be 0..7, got {action_id}")
    vec[action_id] = 1.0

    # ACTION6 carries x/y. No invented coordinates for other actions.
    if action_id == 6:
        if x is None or y is None:
            raise ValueError("ACTION6 requires x and y")
        x = int(np.clip(int(x), 0, 63))
        y = int(np.clip(int(y), 0, 63))
        vec[8] = x / 63.0
        vec[9] = y / 63.0
        if grid is not None:
            arr = np.asarray(grid)
            if arr.ndim == 3:
                arr = arr[-1]
            symbol = int(arr[y, x])
            if 0 <= symbol < 16:
                vec[10 + symbol] = 1.0
    return vec

_ckpt = torch.load(ACTION_MODEL_PATH, map_location="cpu", weights_only=True)
if _ckpt.get("architecture") != "champion_ActionModel_resnet18_backbone":
    raise RuntimeError(f"Unexpected action-model architecture: {_ckpt.get('architecture')!r}")

ACTION_CRITIC = GeneralizedActionModel()
missing, unexpected = ACTION_CRITIC.load_state_dict(_ckpt["model_state_dict"], strict=False)
if missing or unexpected:
    raise RuntimeError(f"Action critic state mismatch; missing={missing}, unexpected={unexpected}")
ACTION_CRITIC.eval()

# Strong structural preflight.
assert ACTION_CRITIC.grid_symbol_embedding.weight.shape == (16, 16)
assert ACTION_CRITIC.action_fc[0].weight.shape[1] == 26
assert ACTION_CRITIC.head_fc[-1].out_features == 1

@torch.inference_mode()
def adl_action_critic_score(grid, action_id, x=None, y=None) -> float:
    arr = np.asarray(grid, dtype=np.int64)
    if arr.ndim == 3:
        arr = arr[-1]
    if arr.shape != (64, 64):
        raise ValueError(f"critic expects settled 64x64 grid, got {arr.shape}")
    feat = encode_action26(action_id, arr, x=x, y=y)
    g = torch.from_numpy(arr).unsqueeze(0)
    a = torch.from_numpy(feat).unsqueeze(0)
    return float(ACTION_CRITIC(g, a).item())

# Write an importable helper for the agent's Python tool environment.
critic_module_path = WORKING_DIR / "adl_differencefusion_critic.py"
critic_module_path.write_text(
    """
# Generated at runtime by ADL DifferenceFusion v2.
# The live notebook process owns the loaded model; this file documents the API contract.
# The ToolAgent should use the in-process `adl_action_critic_score` when available.
ACTION_FEATURE_SCHEMA = '8 action one-hot + x + y + 16 clicked-symbol one-hot'
""".lstrip()
)

print({
    "action_model": str(ACTION_MODEL_PATH),
    "architecture": _ckpt.get("architecture"),
    "training_examples": _ckpt.get("n_training_examples"),
    "source_algorithm": _ckpt.get("source_algorithm"),
    "loss_history": _ckpt.get("loss_history"),
    "action_feature_dim": 26,
})


## 7. DifferenceFusion v2 visible-memory + dual-path + per-move ADL contract

In [ ]:

from inference.agent import tool_agent as _tool_agent

DIFFERENCEFUSION_SENTINEL = "ADL DIFFERENCEFUSION V2 CONTRACT"

DIFFERENCEFUSION_CONTRACT = r"""
ADL DIFFERENCEFUSION V2 CONTRACT — REQUIRED FOR EVERY REAL MOVE

FULL TRACE IS MANDATORY.
Every step below must be emitted as visible assistant text so it appears in the scored
run logs. Never compress several real moves into one unlogged summary.

This is one real ARC-AGI-3 competition trajectory.
Never initialize, fork, clone, replay, or speculatively step a second environment.
Use only evidence from THIS CURRENT GAME and THIS CURRENT RUN.

VISIBLE MEMORY — BEFORE EVERY PYTHON TOOL CALL
Write the complete revised current-game world model in visible assistant text first:
World model:
Goal model:
Action model:
Recent findings:
Open questions:
Plan:
Cross-level notes:

Do not rely on hidden reasoning as persistent memory.

============================================================
PHASE A — EXACTLY TWO CANDIDATES BEFORE EVERY REAL ACTION
============================================================
A = EXPLOIT:
- shortest currently supported progress action
- respect known mechanics
- avoid known no-ops, deaths, loops, and waste

B = EXPLORE:
- highest-information legal action
- target uncertain mechanics or meaningful state change
- avoid exhausted probes

For A and B estimate:
- legality
- predicted progress
- predicted frame change
- information gain
- novelty
- causal consistency
- loop/failure risk
- action efficiency
- current-game world-model consistency
- generalized neural critic value when safely available

Neural critic usage:
- The checkpoint is a GENERIC action-value prior, never a game solution.
- If the Python environment exposes `adl_action_critic_score(grid, action_id, x=None, y=None)`,
  score both candidates on the CURRENT settled 64x64 grid.
- Never fabricate a neural score if the helper is unavailable or an input is uncertain.
- Current-game empirical evidence overrides the generic critic when they conflict.
- Critic influence decays as current-game transition evidence accumulates.

Use the fusion utility:
  progress                0.18
  information_gain        0.13
  predicted_frame_change  0.10
  novelty                 0.10
  causal_consistency      0.12
  neural_critic           0.12
  world_model_consistency 0.10
  action_efficiency       0.08
  minus loop/failure risk 0.07

Before the real action record:
print/emit visibly in the assistant response:
[DIFFERENCEFUSION][PLAN]
DUAL_PATH_DECISION:
STEP=<integer>
A_ACTION=<candidate>
A_PREDICTION=<expected effect>
A_CRITIC=<score or unavailable>
B_ACTION=<candidate>
B_PREDICTION=<expected effect>
B_CRITIC=<score or unavailable>
SELECT=<A or B>
WHY=<short evidence-grounded reason>

Immediately before issuing it, visibly emit:
[DIFFERENCEFUSION][ACTION] STEP=<integer> ACTION=<exact action/data>

Issue exactly ONE selected real environment action.

============================================================
PHASE B — IMMEDIATELY AFTER EVERY REAL ACTION
============================================================
Before planning the next move, compare:
1. state before vs state after
2. predicted state/effect vs actual state/effect
3. critic preference vs actual outcome when a critic score was used

Record visibly before any next plan:
[DIFFERENCEFUSION][RESULT]
POST_MOVE_ADL:
STEP=<same integer>
ACTION=<committed action>
STATE_CHANGED=<yes/no/uncertain>
SCORE_DELTA=<observed or unknown>
LEVEL_DELTA=<observed or unknown>
PREDICTION_MATCH=<yes/partial/no/uncertain>
CRITIC_CALIBRATION=<better/worse/neutral/unavailable>
INFORMATION_GAIN=<0..1>
PROGRESS_VALUE=<-1..1>
LOOP_SIGNAL=<yes/no>
NOVEL_TRANSITION=<yes/no/uncertain>
LESSON=<compact current-game-only causal lesson>
NEXT_BIAS=<exploit/explore/neutral>

The POST_MOVE_ADL lesson MUST change the very next candidate comparison.

============================================================
STRICT NO-PRIOR BOUNDARY
============================================================
Allowed:
- generic Qwen model capability
- generic generalized action critic
- current-game observations/actions/transitions/rewards
- current-game visible world model
- current-game POST_MOVE_ADL records

Forbidden:
- stored winning routes
- prior-game transcripts
- cross-game learned action semantics
- game source-code introspection
- hidden labels
- prior submission solutions
- second-environment rollouts
- cross-game ADL memory

At every new game, reset game-specific ADL memory to empty.
""".strip()

if not getattr(_tool_agent.ToolAgent, "_differencefusion_v2_installed", False):
    _orig_init = _tool_agent.ToolAgent.__init__

    def _differencefusion_init(self, *args, **kwargs):
        _orig_init(self, *args, **kwargs)
        if DIFFERENCEFUSION_SENTINEL not in self._system_prompt:
            self._system_prompt = self._system_prompt.rstrip() + "\n\n" + DIFFERENCEFUSION_CONTRACT

    _tool_agent.ToolAgent.__init__ = _differencefusion_init
    _tool_agent.ToolAgent._differencefusion_v2_installed = True

# Qwen 3.8 analyzer, preserving TAAF's normal tool transport.
class DifferenceFusionToolAgent(_tool_agent.ToolAgent):
    pass

def _differencefusion_analyzer_factory(game, index):
    model = (
        os.environ.get("INFERENCE_ANALYZER_MODEL")
        or os.environ.get("LOCAL_ANALYZER_MODEL_ID")
        or "saltb0x/Qwen3.8-27B-FP8"
    )
    base_url = (
        os.environ.get("LOCAL_ANALYZER_BASE_URL")
        or os.environ.get("OPENAI_BASE_URL")
        or "http://127.0.0.1:1234/v1"
    )
    return DifferenceFusionToolAgent(
        model=model,
        timeout=bm.solver.analyzer_timeout,
        save_request_logs=bm.solver.save_request_logs,
        base_url=base_url,
        provider="vllm",
    )

bm.solver.analyzer_factory = _differencefusion_analyzer_factory

_probe = DifferenceFusionToolAgent(model=getattr(bm.solver, "model", "local"))
for marker in [
    DIFFERENCEFUSION_SENTINEL,
    "DUAL_PATH_DECISION:",
    "POST_MOVE_ADL:",
    "generalized neural critic",
    "Never initialize, fork, clone, replay",
]:
    if marker not in _probe._system_prompt:
        raise RuntimeError(f"DifferenceFusion prompt preflight missing: {marker}")

print("ADL DifferenceFusion v2 contract installed.", flush=True)


## 8. Discover live competition games and execute exactly one scored pass

In [ ]:

def _wait_for_gateway(base_url: str, timeout_s: float = 600.0):
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle competition gateway did not become ready: {last_error}")

def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]

print((BUNDLE_DIR / "preamble.txt").read_text())
if (BUNDLE_DIR / "git_status.txt").exists():
    (WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))
os.environ.setdefault("ARC_API_KEY", "test-key-123")
os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")

_wait_for_gateway(os.environ["ARC_BASE_URL"])
bm.games = _competition_games()
bm.n_passes = 1
bm.game_weights = None

print("[DIFFERENCEFUSION][RUN] ACTUAL SCORED RUN START", flush=True)
print(
    f"ACTUAL SCORED RUN START games={len(bm.games)} "
    f"passes={bm.n_passes} concurrency={bm.solver.concurrency}",
    flush=True,
)

run_error = None
try:
    await bm.run(
        soft_end_time=None,
        runtime_environment=target,
        minimal_diagnostics=False,
    )
except Exception as exc:
    run_error = exc
    raise
finally:
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print("TEARDOWN:", command, flush=True)
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )


## 9. Validate the real competition submission and audit ADL coverage

In [ ]:

import re

submission_path = WORKING_DIR / "submission.parquet"
if not submission_path.exists():
    raise RuntimeError(
        "Actual competition rerun completed without submission.parquet. "
        "No dummy/fallback submission is permitted."
    )

if submission_path.stat().st_size <= 0:
    raise RuntimeError("submission.parquet exists but is empty.")

_MARKERS = ("DUAL_PATH_DECISION:", "POST_MOVE_ADL:")
_TEXT_EXTS = {".log", ".txt", ".json", ".jsonl", ".md"}

def _scan_adl_markers(root: Path):
    hits = []
    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in _TEXT_EXTS:
            continue
        if path.name in {"differencefusion_v2_audit.json"}:
            continue
        try:
            text = path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue
        for marker in _MARKERS:
            count = text.count(marker)
            if count:
                hits.append({"file": str(path), "marker": marker.rstrip(":"), "count": count})
    return hits

hits = _scan_adl_markers(WORKING_DIR)
post_move_count = sum(h["count"] for h in hits if h["marker"] == "POST_MOVE_ADL")
decision_count = sum(h["count"] for h in hits if h["marker"] == "DUAL_PATH_DECISION")

total_actions = sum(
    len(getattr(run, "history", ()) or ())
    for run in getattr(bm, "game_runs", [])
)

audit = {
    "schema": "adl.arc3.differencefusion-v2.scored-run.v1",
    "competition_rerun": True,
    "full_trace_required": True,
    "minimal_diagnostics": False,
    "embedded_action_model": False,
    "games": len(getattr(bm, "game_runs", []) or []),
    "total_actions": total_actions,
    "dual_path_decisions": decision_count,
    "post_move_adl_updates": post_move_count,
    "post_move_coverage": post_move_count / total_actions if total_actions else 0.0,
    "single_environment_pass": int(getattr(bm, "n_passes", 0) or 0) == 1,
    "submission_path": str(submission_path),
    "submission_bytes": submission_path.stat().st_size,
    "neural_critic_checkpoint": str(ACTION_MODEL_PATH),
    "qwen_model": "saltb0x/Qwen3.8-27B-FP8",
    "strict_no_prior": True,
    "hits": hits,
}

audit_path = WORKING_DIR / "differencefusion_v2_audit.json"
audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n")

print(json.dumps(audit, indent=2))
print("REAL SUBMISSION:", submission_path)
